# CaseForge: Phase 2 — LLM Fine-Tuning
### Fine-tuning Llama 2 7B with LoRA on 74 IFQM-aligned business case studies
**For: IFQM Bangalore & SRM Q Club | Submitted by Viva Baranwal**

> Before running: Upload `data/training_data.json` and `data/embeddings/` to a Google Drive folder called `Case IQ/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base_path = '/content/drive/My Drive/CaseForge'
print("Files in CaseForge folder:")
print(os.listdir(base_path))

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate torch
!pip install -q sentence-transformers chromadb
print("All dependencies installed.")

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
import json

with open(f'{base_path}/data/training_data.json') as f:
    all_cases = json.load(f)

print(f"Total cases loaded: {len(all_cases)}")
print(f"Example company: {all_cases[0].get('company', all_cases[0].get('organization', 'N/A'))}")
print(f"Keys in each case: {list(all_cases[0].keys())}")

In [ ]:
training_messages = []

for case in all_cases:
    # Handle both 'company' and 'organization' keys since parse_cases.py uses organization
    company = case.get('company') or case.get('organization') or 'Unknown Company'
    industry = case.get('industry', 'Business')
    background = case.get('background') or case.get('setting') or ''
    challenge = case.get('challenge') or case.get('themes') or case.get('crisis') or ''
    intervention = case.get('intervention') or case.get('solution') or ''
    results = case.get('results') or case.get('outcomes') or ''
    learning = case.get('learning_outcomes') or case.get('learning') or ''

    # Skip incomplete cases
    if not background or not challenge:
        continue

    message = {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert academic case writer aligned with IFQM standards. Given company data, generate a structured teaching case study starting with an executive protagonist's dilemma. Sections: BACKGROUND, THEMES, INTERVENTION, RESULTS, LEARNING OUTCOMES. A case study describes what happened \u2014 never prescribe. Analysis belongs to the student."
            },
            {
                "role": "user",
                "content": f"Company: {company}\nIndustry: {industry}\nGenerate a complete IFQM-aligned case study."
            },
            {
                "role": "assistant",
                "content": f"BACKGROUND:\n{background}\n\nTHEMES:\n{challenge}\n\nINTERVENTION:\n{intervention}\n\nRESULTS:\n{results}\n\nLEARNING OUTCOMES:\n{learning}"
            }
        ]
    }
    training_messages.append(message)

print(f"Valid training examples: {len(training_messages)} out of {len(all_cases)} cases")

# Save formatted data to Drive
with open(f'{base_path}/training_formatted.json', 'w') as f:
    json.dump(training_messages, f)
print("Saved training_formatted.json to Drive.")

## Step 2: Download and Load Llama 2 7B
\u23f1\ufe0f This takes 10\u201315 minutes (downloading ~13 GB). Do not close the tab.

You need a Hugging Face account and must accept the Llama 2 license at:
https://huggingface.co/meta-llama/Llama-2-7b-hf

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-2-7b-hf"

# Login to Hugging Face (paste your token when prompted)
from huggingface_hub import login
login()

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"Model loaded: {model_name}")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,                                    # Rank \u2014 16 is good balance of quality vs speed
    lora_alpha=32,                           # Scaling factor
    target_modules=["q_proj", "v_proj"],     # Train only attention layers
    lora_dropout=0.05,                       # Regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Expected output:
# trainable params: ~4,194,304 || all params: ~6,738,415,616 || trainable%: 0.06

In [ ]:
from datasets import Dataset
import json

with open(f'{base_path}/training_formatted.json') as f:
    training_data = json.load(f)

def format_for_training(example):
    messages = example['messages']
    text = ""
    for msg in messages:
        if msg['role'] == 'system':
            text += f"<s>[INST] <<SYS>>\n{msg['content']}\n<</SYS>>\n\n"
        elif msg['role'] == 'user':
            text += f"{msg['content']} [/INST] "
        elif msg['role'] == 'assistant':
            text += f"{msg['content']} </s>"
    return {"text": text}

raw_dataset = Dataset.from_list(training_data)
dataset = raw_dataset.map(format_for_training)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")

## Step 3: Train the Model
\u26a0\ufe0f This cell takes 4\u20136 hours on a free Colab GPU (T4).

Do NOT close this tab. Loss should decrease over time \u2014 if it goes up, something is wrong.

Target final loss: **below 1.0**

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=f"{base_path}/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,        # Keep at 2 for T4 GPU memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,        # Effective batch size = 8
    learning_rate=2e-4,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,                   # Keep only last 2 checkpoints to save Drive space
    logging_steps=25,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    fp16=True,                            # Use float16 for T4 GPU
    report_to="none"                      # Disable wandb
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
save_path = f"{base_path}/final_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print(f"Files saved: {os.listdir(save_path)}")

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the saved model
test_tokenizer = AutoTokenizer.from_pretrained(f"{base_path}/final_model")
test_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)
test_model = PeftModel.from_pretrained(test_model, f"{base_path}/final_model")

# Run a test prompt
prompt = """<s>[INST] <<SYS>>
You are an expert academic case writer aligned with IFQM standards.
<</SYS>>

Company: Wipro
Industry: IT Services
Generate a complete IFQM-aligned case study. [/INST]"""

inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

result = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated case study preview:")
print(result[len(prompt):])

## \u2705 Training Complete

### Next Steps:
1. Download `final_model/` folder from Google Drive
2. Place it at: `project_live/model/final_model/`
3. Run the backend:
```bash
cd project_live/backend
uvicorn app:app --reload
```
4. Open `project_live/index.html` in browser
5. Fill the 9-screen wizard and generate your first real case study

### If Colab session times out mid-training:
- Checkpoints are saved every 100 steps to `CaseForge/checkpoints/`
- Resume by loading the latest checkpoint in Cell 12:
  - Add `resume_from_checkpoint=True` to `trainer.train()`